<a href="https://colab.research.google.com/github/Salman-id85/AI-Based-Salary-Prediction-System-for-HR-Analytics/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.9 MB/s eta 0:00:00


In [2]:
from pyngrok import ngrok

# Set your authtoken
!ngrok config add-authtoken 24YIhUmElXYMQewJorZK3fJC8aB_TFVvBse7m4mV2eMNCTos


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
!pip install streamlit pyngrok --quiet


In [8]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url)
df.head()


,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
0,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
1,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
2,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
3,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
4,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


In [9]:
import pandas as pd

# Load dataset from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

# Assign column names manually
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain",
    "capital_loss", "hours_per_week", "native_country", "salary"
]

df = pd.read_csv(url, names=columns, sep=',', skipinitialspace=True)
df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [10]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Drop rows with missing values
df = df.dropna()

# Label encode education, occupation, salary
le_edu = LabelEncoder()
le_occ = LabelEncoder()
le_sal = LabelEncoder()

df['education'] = le_edu.fit_transform(df['education'])
df['occupation'] = le_occ.fit_transform(df['occupation'])
df['salary'] = le_sal.fit_transform(df['salary'])  # '>50K' becomes 1, '<=50K' becomes 0

# Select features
X = df[['age', 'education', 'occupation', 'hours_per_week']]
y = df['salary']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Save model and encoders
joblib.dump(model, "salary_model.pkl")
joblib.dump(le_edu, "le_edu.pkl")
joblib.dump(le_occ, "le_occ.pkl")


['le_occ.pkl']

In [11]:

%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load trained model and encoders
model = joblib.load("salary_model.pkl")
le_edu = joblib.load("le_edu.pkl")
le_occ = joblib.load("le_occ.pkl")

st.title("💼 Employee Salary Prediction App")
st.markdown("Predict whether an employee earns **>50K** or **<=50K** using ML")

# User Inputs
age = st.slider("Age", 18, 70, 30)
education = st.selectbox("Education", le_edu.classes_)
occupation = st.selectbox("Occupation", le_occ.classes_)
hours_per_week = st.slider("Hours per Week", 1, 99, 40)

# Predict
if st.button("Predict Salary Class"):
    edu_encoded = le_edu.transform([education])[0]
    occ_encoded = le_occ.transform([occupation])[0]

    input_data = pd.DataFrame({
        'age': [age],
        'education': [edu_encoded],
        'occupation': [occ_encoded],
        'hours_per_week': [hours_per_week]
    })

    prediction = model.predict(input_data)[0]
    result = ">50K" if prediction == 1 else "<=50K"
    st.success(f"💰 Predicted Salary Class: {result}")


Writing app.py


In [16]:
!ngrok config add-authtoken 24YIhUmElXYMQewJorZK3fJC8aB_TFVvBse7m4mV2eMNCTos

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [17]:
from pyngrok import ngrok

# Start new ngrok tunnel
public_url = ngrok.connect(8501)
print("🔗 App URL:", public_url)

# Launch Streamlit in background
!streamlit run app.py &>/content/log.txt &


🔗 App URL: NgrokTunnel: "https://6400641f0674.ngrok-free.app" -> "http://localhost:8501"
